# Perth Property Price Prediction - Model Training

**Author:** Haz Li
**Date:** August, 2025

## Objective
To develop a robust machine learning model that accurately predicts Perth property prices. This notebook outlines the complete, end-to-end workflow from data loading to model serialization, using a clear and proven methodology.

In [31]:
# --- 1. Import Necessary Libraries ---

import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
import joblib  # For saving our model and columns
import warnings
import os
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

print("Libraries imported successfully.")

Libraries imported successfully.


## 2. Data Loading from MySQL

We connect directly to our structured MySQL database to ensure we are working with the clean, canonical data source established in the ETL phase. The query joins all necessary tables to create a comprehensive, flat dataset for modeling.

In [32]:
# --- 2. Database Connection and Data Loading  ---
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv()
print("Loading environment variables...")

DB_USER = os.getenv("DB_USER", "root")
DB_PASS = os.getenv("DB_PASS")

DB_HOST = '127.0.0.1' # Use the loopback IP address to avoid encoding issues
# ======================================================================
DB_PORT = '3306'
DB_NAME = 'perth_property_db'

if not DB_PASS:
    raise ValueError("DB_PASS not found in .env file.")

# Create the final, robust connection string
db_connection_str = f'mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
engine = create_engine(db_connection_str)

# The definitive query to get all features needed for the model
query = """
    SELECT
        p.price, p.land_size, p.parking_spaces, p.distance_to_cbd,
        p.property_type, s.suburb_name, l.bedrooms, l.bathrooms,
        ps.primary_school_icsea, ss.secondary_school_icsea
    FROM FACT_Properties p
    JOIN DIM_Suburbs s ON p.suburb_id = s.suburb_id
    JOIN DIM_Layouts l ON p.layout_id = l.layout_id
    LEFT JOIN DIM_Primary_Schools ps ON p.primary_school_id = ps.primary_school_id
    LEFT JOIN DIM_Secondary_Schools ss ON p.secondary_school_id = ss.secondary_school_id;
"""

print("Loading data from database...")
df = pd.read_sql(text(query), engine)
print(f"Data loaded successfully with shape: {df.shape}")
display(df.head())

Loading environment variables...
Loading data from database...
Data loaded successfully with shape: (42954, 10)


,price,land_size,parking_spaces,distance_to_cbd,property_type,suburb_name,bedrooms,bathrooms,primary_school_icsea,secondary_school_icsea
0,395000.0,411,1,13800,duplex-semi-detached,Alexander Heights,3,1,996,977
1,625000.0,695,2,14299,house,Alexander Heights,6,3,994,1030
2,275000.0,181,1,13940,villa,Alexander Heights,3,1,994,1030
3,455000.0,547,2,14609,house,Alexander Heights,4,2,994,1030
4,450000.0,714,3,13816,house,Alexander Heights,4,2,994,1010


## 3. Feature Engineering & Preparation

This is the most critical stage. We will select our final features and transform them into a format suitable for the machine learning model.

- **Feature Set:** We will use a comprehensive set of features including physical, geographical, and environmental attributes. Crucially, we will use `suburb_name` directly to capture fine-grained location value.
- **Handling Missing Values:** Any residual missing values (e.g., for school scores) will be imputed using the median.
- **One-Hot Encoding:** All categorical features (`suburb_name`, `property_type`) will be converted into numerical format using one-hot encoding.

In [ ]:
# --- 3. Final Feature Engineering and Preparation ---

# --- 3a. Handle Missing Values ---
df['primary_school_icsea'].fillna(df['primary_school_icsea'].median(), inplace=True)
df['secondary_school_icsea'].fillna(df['secondary_school_icsea'].median(), inplace=True)
df.dropna(inplace=True)
print("Missing values handled.")

# --- 3b. Feature Selection ---
# This is our final, definitive list of features.
features_to_use = [
    'bedrooms', 
    'bathrooms', 
    'land_size', 
    'parking_spaces', 
    'primary_school_icsea',
    'secondary_school_icsea',
    'suburb_name',      # The most important fine-grained location feature
    'property_type'
]
target = 'price'

df_model = df[features_to_use + [target]].dropna()

# --- 3c. Define Explicit Categorical Types ---
# First, find all unique categories from our training data
suburb_categories = df_model['suburb_name'].unique()
property_type_categories = df_model['property_type'].unique()

# Then, convert the columns to Pandas' special Categorical type
df_model['suburb_name'] = pd.Categorical(df_model['suburb_name'], categories=suburb_categories)
df_model['property_type'] = pd.Categorical(df_model['property_type'], categories=property_type_categories)

# Save these category lists for the web app!
joblib.dump(suburb_categories, 'suburb_categories.pkl')
joblib.dump(property_type_categories, 'property_type_categories.pkl')
print("\nSaved suburb and property type category lists successfully.")
# ======================================================================

# --- 3d. One-Hot Encoding (Now it will use the defined categories) ---
df_model_encoded = pd.get_dummies(df_model, columns=['suburb_name', 'property_type'], prefix=['suburb', 'type'], drop_first=True)
print(f"Data prepared for modeling. Shape after encoding: {df_model_encoded.shape}")

# --- 3e. Separate Features (X) and Target (y) ---
X = df_model_encoded.drop(target, axis=1)
y = df_model_encoded[target]

print(f"\nFinal shape of X (features): {X.shape}")
print(f"Final shape of y (target): {y.shape}")

Missing values handled.

Saved suburb and property type category lists successfully.
Data prepared for modeling. Shape after encoding: (42954, 219)

Final shape of X (features): (42954, 218)
Final shape of y (target): (42954,)


## 4. Model Training & Evaluation

The data is now ready. We will split it into training and testing sets, train a RandomForestRegressor model, and evaluate its performance on the unseen test data using the R-squared (R²) metric.

In [34]:
# --- 4. Train/Test Split ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training set: {X_train.shape[0]} samples | Testing set: {X_test.shape[0]} samples")

# --- 4b. Initialize and Train the Model ---
print("\nTraining the RandomForestRegressor model...")
final_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1, oob_score=True)
final_model.fit(X_train, y_train)
print("Model training complete.")

# --- 4c. Evaluate the Model ---
print(f"\nModel Out-of-Bag (OOB) Score: {final_model.oob_score_:.4f}")
y_pred = final_model.predict(X_test)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Test Set R-squared (R²) Score: {r2:.4f}")
print(f"Test Set Root Mean Squared Error (RMSE): ${rmse:,.2f}")

# --- 4d. Feature Importance ---
print("\n--- Top 15 Most Important Features ---")
feature_importances = pd.Series(final_model.feature_importances_, index=X.columns).sort_values(ascending=False)
display(feature_importances.head(15))

Training set: 34363 samples | Testing set: 8591 samples

Training the RandomForestRegressor model...
Model training complete.

Model Out-of-Bag (OOB) Score: 0.7853
Test Set R-squared (R²) Score: 0.8023
Test Set Root Mean Squared Error (RMSE): $257,135.34

--- Top 15 Most Important Features ---


primary_school_icsea       0.304256
bathrooms                  0.227492
land_size                  0.183286
distance_to_cbd            0.079630
secondary_school_icsea     0.042606
bedrooms                   0.030618
suburb_Cottesloe           0.024692
parking_spaces             0.021400
suburb_Mount Pleasant      0.006156
suburb_Peppermint Grove    0.004544
suburb_City Beach          0.003937
suburb_North Coogee        0.003883
suburb_Swanbourne          0.002963
suburb_Mosman Park         0.002826
suburb_North Beach         0.002349
dtype: float64

## 5. Model Serialization

The final, trained model and its required column structure are saved to disk using `joblib`. These files are the essential assets for deployment in our Flask web application.

In [35]:
# --- 5. Save the Model and Columns for Deployment ---
model_columns = X.columns
joblib.dump(final_model, 'property_price_predictor.pkl')
joblib.dump(model_columns, 'model_columns.pkl')
print("\nModel and columns saved successfully and are ready for deployment.")


Model and columns saved successfully and are ready for deployment.
